# Land Cover Mapping Methods Comparison

## Pre-requisistes
- Earth Engine Initialization
- AOI retrieval

In [1]:
import ee
import luma_ge
ee.Authenticate() #force=True use for re-authentication
ee.Initialize()


Successfully saved authorization token.


In [ ]:
#############################  Area of Interest  ###########################
def get_aoi_from_gaul(country="Indonesia", province="Sumatera Selatan"):
    """
    Get Area of Interest geometry from GAUL administrative boundaries.
    
    Parameters:
    -----------
    country : str
        Country name (default: "Indonesia")
    province : str
        Province/state name (default: "Sumatera Selatan")
        
    Returns:
    --------
    ee.Geometry : Area of interest geometry
    """
    admin = ee.FeatureCollection("FAO/GAUL/2015/level1")
    aoi_fc = admin.filter(ee.Filter.eq('ADM0_NAME', country)).filter(
        ee.Filter.eq('ADM1_NAME', province)
    )
    return aoi_fc.geometry()
aoi = get_aoi_from_gaul()


## Direct Classification
1. Search The Imagery 
2. Define the predictor
3. Run the classification

In [2]:
#import the library
import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image
aoi = geemap.shp_to_ee('C:/Data Spasial/Adaptive workflow testing/GPS-points/AOI_Oganilir.shp')
#initilize reflectance data retrieval class
reflectance = Reflectance_Data()
#retrieve the image collection
#use 2016-2017 image for better coverage
img_col, stat = reflectance.get_optical_data(aoi, #Aoi
                                       '2017-01-01', '2017-12-31', #start, end
                                       optical_data='L8_SR', #sensor
                                       cloud_cover=50, #cloud cover
                                       compute_detailed_stats=False)
#get the thermal band
thermal, stat = reflectance.get_thermal_bands(aoi, 
                                              '2017-01-01', '2017-12-31',
                                              thermal_data='L8_TOA',
                                              cloud_cover=50,
                                              compute_detailed_stats=False)
#initilize the compositing class
comp = final_Image()
#create the composite for multispectral and thermal data
med_landsat = comp.get_temporal_composite(img_col, aoi, reducer='Median')
median_landsat = med_landsat.select(
    med_landsat.bandNames().remove('AEROSOL')
)
thermal_median = comp.get_temporal_composite(thermal, aoi )
#define the visulization parameter and show them on the map
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
map = geemap.Map()
map.centerObject(aoi, zoom=10)
map.addLayer(thermal_median, {}, "Thermal")
map.addLayer(img_col,l8_sr_visparam, "Collection")
map.addLayer(median_landsat, l8_sr_visparam, "Median Image")
map

c:\Users\AFahrezi\AppData\Local\anaconda3\envs\luma-lite\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-21 13:58:34,272 - luma_ge.ee_config - INFO - Earth Engine initialized successfully
2026-04-21 13:58:34,273 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-04-21 13:58:34,274 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-04-21 13:58:34,274 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-04-21 13:58:34,275 - Reflectance_Data - INFO - Cloud cover threshold: 50%
2026-04-21 13:58:34,275 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-04-21 13:58:34,277 - Reflectance_Stats - INFO - Reflectance Stats i

Map(center=[-3.4152616959981676, 104.60534276364243], controls=(WidgetControl(options=['position', 'transparen…

## Terrain and Spectral Predictors


In [24]:
from luma_ge.predictor import terrain_calculator, SpectralCalculator
# Initialize predictor backends
terrain_calc = terrain_calculator()
spectral_calc = SpectralCalculator()

# Choose DEM source and compute terrain layers
dem_source = 'NASADEM'
elevation = terrain_calc.calculate_elevation(aoi, dem_source=dem_source)
slope = terrain_calc.calculate_slope(aoi, dem_source=dem_source)
aspect = terrain_calc.calculate_aspect(aoi, dem_source=dem_source)

# Visualize terrain layers on the map
terrain_vis = {
    'min': 0,
    'max': 500,
    'palette': ['black', 'blue', 'green', 'yellow', 'red']
}
slope_vis = {
    'min': 0,
    'max': 45,
    'palette': ['white', 'lightblue', 'green', 'yellow', 'red']
}
aspect_vis = {
    'min': 0,
    'max': 360,
    'palette': ['white', 'blue', 'green', 'yellow', 'red', 'purple']
}

map.addLayer(elevation, terrain_vis, f"Elevation ({dem_source})")
map.addLayer(slope, slope_vis, f"Slope ({dem_source})")
#map.addLayer(aspect, aspect_vis, f"Aspect ({dem_source})")

# Compute spectral indices from the optical collection
indices_to_compute = ['EVI', 'GBNDVI', 'MSAVI', 'NDVI', 'MNDWI', 'AWEInsh', 'NDBI', 'NDMI', 'MBI' ]
spectral_indices = spectral_calc.calculate_indices_with_collection(
    collection=img_col,
    aoi=aoi,
    index_list=indices_to_compute,
    reducer_method='mean'
)

print('Computed terrain layers: elevation, slope, aspect')
print('Computed spectral indices:', spectral_indices.bandNames().getInfo() if spectral_indices else 'failed')

if spectral_indices:
    ndvi_vis = {'min': -1, 'max': 1, 'palette': ['purple', 'white', 'green']}
    evi_vis = {'min': -1, 'max': 1, 'palette': ['navy', 'white', 'lime']}
    map.addLayer(spectral_indices.select('GBNDVI'), ndvi_vis, 'GBNDVI')
    map.addLayer(spectral_indices.select('EVI'), evi_vis, 'EVI')
#stack the predictors and the imagery
predictor_stack = median_landsat.addBands(thermal_median).addBands(spectral_indices).addBands(elevation).addBands(slope).toFloat()
print('Final predictor stack bands:', predictor_stack.bandNames().getInfo() if predictor_stack else 'failed')


2026-04-21 14:28:10,710 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-04-21 14:28:10,711 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-04-21 14:28:10,712 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-04-21 14:28:10,712 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-04-21 14:28:10,713 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-04-21 14:28:10,713 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-04-21 14:28:10,714 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-04-21 14:28:10,715 - luma_ge.predictor - INFO - Successfully calculated slope layer using NASADEM DEM
2026-04-21 14:28:10,715 - luma_ge.predictor - INFO - Calculating aspect layer using NASADEM DEM...
2026-04-21 14:28:10,716 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...


Computed terrain layers: elevation, slope, aspect
Computed spectral indices: ['EVI', 'GBNDVI', 'MSAVI', 'NDVI', 'MNDWI', 'AWEInsh', 'NDBI', 'NDMI', 'MBI']
Final predictor stack bands: ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'THERMAL', 'EVI', 'GBNDVI', 'MSAVI', 'NDVI', 'MNDWI', 'AWEInsh', 'NDBI', 'NDMI', 'MBI', 'elevation', 'slope']


## Classification 

In [25]:
#extract the pixel sample
from luma_ge.classification import FeatureExtraction, Generate_LULC
#samples
sample = geemap.shp_to_ee('C:/Data Spasial/Adaptive workflow testing/GPS-points/GT_2016/GT_2016_epistem_OganIlir.shp')
features = FeatureExtraction()
train, test = features.stratified_split(sample, predictor_stack, 
                            class_prop='ID_epistem', train_ratio=0.7, seed=42)

2026-04-21 14:28:58,716 - pyogrio._io - INFO - Created 700 records


Stratified Random Split Training Pixel Size: 503
Stratified Random Split Testing Pixel Size: 197


In [26]:
#initilizae the classification class
clf = Generate_LULC()
#applied hard classification/original MS
classification_raw, model_raw = clf.hard_classification(training_data = train, 
 class_property='ID_epistem',
 image=predictor_stack, 
 ntrees=350,
 v_split=9,
 return_model=True)
#Land cover class definition
lc_class = {
    1: {"name": "Secondary Dryland Forest",   "color": "#054504"},
    6: {"name": "Secondary Swamp Forest",   "color": "#059486"},
    16: {"name": "Mixed/home Garden", "color": "#0deb50"},
    15: {"name": "Rubber Agroforest",    "color": "#839248"},
    #14: {"name": "Coffee Agroforest",    "color": "#df980a"},
    #7: {"name": "Plantation Forest",    "color": "#09a726"},
    9: {"name": "Oil Palm Monoculture",    "color": "#d9cc66"},
    8: {"name": "Rubber monoculture",    "color": "#414127"},
    11: {"name": "Coconut monoculture",    "color": "#d9e66c"},
    12: {"name": "Other monoculture",    "color": "#6aa66d"},
    17: {"name": "Paddy Field",    "color": "#b1eb03"},
    13: {"name": "Other Cropland",    "color": "#f1d900"},
    18: {"name": "Grass or Savanna",    "color": "#bdf2c0"},    
    21: {"name": "Cleared land",    "color": "#413d2f"},
    20: {"name": "Settlement",    "color": "#e00c0c"},
    #24: {"name": "Fish Pond",    "color": "#e18adb"},
    23: {"name": "Water body",    "color": "#0b3bdb"},                  
}
ogan_ilir_class = {
    1: {"name": "Secondary Dryland Forest",   "color": "#054504"},
    6: {"name": "Secondary Swamp Forest",   "color": "#059486"},
    8: {"name": "Rubber monoculture",    "color": "#583405"},
    9: {"name": "Oil Palm Monoculture",    "color": "#e0a911"},
    13: {"name": "Other Cropland",    "color": "#f1d900"},
    15: {"name": "Rubber Agroforest",    "color": "#839248"},
    16: {"name": "Mixed/home Garden", "color": "#0deb50"},
    17: {"name": "Paddy Field",    "color": "#b1eb03"},
    18: {"name": "Grass or Savanna",    "color": "#bdf2c0"},
    20: {"name": "Settlement",    "color": "#e00c0c"},
    21: {"name": "Cleared land",    "color": "#464440"},
    23: {"name": "Water body",    "color": "#0b3bdb"},
}

In [27]:
#Evaluate model performance
print("Evaluating raw classification.")
try:
    model_acc_first = clf.evaluate_model(
        trained_model=model_raw,
        test_data=test,
        class_property='ID_epistem'
    )
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    
import pandas as pd
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {model_acc_first['overall_accuracy']:.4f} ({model_acc_first['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {model_acc_first['kappa']:.4f}")
print(f"Overall G-Mean: {model_acc_first['overall_gmean']:.4f}")
print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_df = pd.DataFrame({
    'Precision': model_acc_first['precision'],
    'Recall': model_acc_first['recall'],
    'F1-Score': model_acc_first['f1_scores'],
    'G-Mean': model_acc_first['gmean_per_class']
})
#Round to 4 decimal places
metrics_df = metrics_df.round(4)
display(metrics_df)

Evaluating raw classification.
=== Model Performance Summary ===
Overall Accuracy: 0.6786 (67.86%)
Kappa Coefficient: 0.6415
Overall G-Mean: 0.5086

=== Per-Class Metrics ===


,Precision,Recall,F1-Score,G-Mean
0,0.0000,0.0000,0.0000,0.0000
1,0.7778,0.5833,0.6667,0.6736
2,0.0000,0.0000,0.0000,0.0000
3,0.0000,0.0000,0.0000,0.0000
4,0.0000,0.0000,0.0000,0.0000
5,0.0000,0.0000,0.0000,0.0000
6,0.7368,0.7000,0.7179,0.7182
7,0.0000,0.0000,0.0000,0.0000
8,0.4000,0.6250,0.4878,0.5000
9,0.7500,0.8750,0.8077,0.8101


Overall Accuracy: 0.4561 (45.61%)
Kappa Coefficient: 0.3600

In [23]:
#visualize the feature importance
import plotly.express as px
importance = clf.get_feature_importance(model_raw, training_data=train, class_property='ID_epistem')
fig = px.bar(
                importance,
                x='Importance',
                y='Band',
                orientation='h',
                title='Kanal mana yang paling penting?',
                color='Importance',
                color_continuous_scale='Viridis',
                text='Importance'
            )
            
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(
                yaxis={'categoryorder': 'total ascending'},
                height=max(400, len(importance) * 30),
                showlegend=False
            )
            
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Importance=%{marker.color}<br>Band=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': {'bdata': ('Vl1jisoShECbw4bpQDaAQHD0+Ox4QH' ... 'eviHpAxQCyZscpeUDiiEk9HT54QA=='),
                                   'dtype': 'f8'},
                         'coloraxis': 'coloraxis',
                         'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'text': {'bdata': ('Vl1jisoShECbw4bpQDaAQHD0+Ox4QH' ... 'eviHpAxQCyZscpeUDiiEk9HT54QA=='),
                       'dtype': 'f8'},
              'textposition': 'outside',
              'texttemplate': '%{text:.3f}',
              'type': 'bar',
              'x': {'bdata': ('Vl1jisoShECbw4bpQDaAQHD0+Ox4QH' ... 'eviHpAxQCyZscpeUDiiEk9HT54QA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['elevation', 'AWEInsh', 'MNDWI', 'RED', 'GREEN', 'NIR', 'SWIR1',
                          'THERMAL', 'MBI', 'BLUE', 'NDVI', 'GBNDVI', 'SWIR2', 'EVI', 'NDBI',
                          'NDMI', 'MSAVI', 'slope', 'SAVI', 'aspect'], dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'coloraxis': {'colorbar': {'title': {'text': 'Importance'}},
                             'colorscale': [[0.0, '#440154'], [0.1111111111111111,
                                            '#482878'], [0.2222222222222222,
                                            '#3e4989'], [0.3333333333333333,
                                            '#31688e'], [0.4444444444444444,
                                            '#26828e'], [0.5555555555555556,
                                            '#1f9e89'], [0.6666666666666666,
                                            '#35b779'], [0.7777777777777778,
                                            '#6ece58'], [0.8888888888888888,
                                            '#b5de2b'], [1.0, '#fde725']]},
               'height': 600,
               'legend': {'tracegroupgap': 0},
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Kanal mana yang paling penting?'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Importance'}},
               'yaxis': {'anchor': 'x',
                         'categoryorder': 'total ascending',
                         'domain': [0.0, 1.0],
                         'title': {'text': 'Band'}}}
})

In [28]:
def get_sat_embedding(aoi, start_year, end_year):
     """
    Retrieves the satellite embedding data
    for the specified time range and area of interest (AOI).
    Returns:
        ee.Image: A median-composited, clipped image.
    """
     dataset = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

     image = dataset\
        .filterDate(f'{start_year}-01-01', f'{end_year}-01-01') \
         .filterBounds(aoi)
     mosaicked = image.mosaic().clip(aoi)
     return mosaicked
#get the satellite embedding data
data_2020 = get_sat_embedding(aoi, 2017, 2018)


In [29]:
#extract the pixel from embedding data
train_embed, test_embed = features.stratified_split(sample, data_2020, 
                            class_prop='ID_epistem', train_ratio=0.7, seed=42)
#applied hard classification/original MS
classification_embed, model_embed = clf.hard_classification(training_data = train_embed, #Ms only training data
 class_property='ID_epistem', 
 image=data_2020, 
 ntrees=300,
 v_split=13,
 return_model=True)

Stratified Random Split Training Pixel Size: 503
Stratified Random Split Testing Pixel Size: 197


In [30]:
#Evaluate model performance
print("Evaluating embedding classification.")
try:
    model_acc_embed = clf.evaluate_model(
        trained_model=model_embed,
        test_data=test_embed,
        class_property='ID_epistem'
    )
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    
import pandas as pd
# Display accuracy results
print("=== Model Performance Summary ===")
print(f"Overall Accuracy: {model_acc_embed['overall_accuracy']:.4f} ({model_acc_embed['overall_accuracy']*100:.2f}%)")
print(f"Kappa Coefficient: {model_acc_embed['kappa']:.4f}")
print(f"Overall G-Mean: {model_acc_embed['overall_gmean']:.4f}")
print("\n=== Per-Class Metrics ===")
#Class Dataframe
metrics_embed = pd.DataFrame({
    'Precision': model_acc_embed['precision'],
    'Recall': model_acc_embed['recall'],
    'F1-Score': model_acc_embed['f1_scores'],
    'G-Mean': model_acc_embed['gmean_per_class']
})
#Round to 4 decimal places
metrics_embed = metrics_embed.round(4)
display(metrics_embed)

Evaluating embedding classification.
=== Model Performance Summary ===
Overall Accuracy: 0.7614 (76.14%)
Kappa Coefficient: 0.7345
Overall G-Mean: 0.7478

=== Per-Class Metrics ===


,Precision,Recall,F1-Score,G-Mean
0,0.0000,0.0000,0.0000,0.0000
1,0.7500,0.7500,0.7500,0.7500
2,0.0000,0.0000,0.0000,0.0000
3,0.0000,0.0000,0.0000,0.0000
4,0.0000,0.0000,0.0000,0.0000
5,0.0000,0.0000,0.0000,0.0000
6,0.7000,0.7000,0.7000,0.7000
7,0.0000,0.0000,0.0000,0.0000
8,0.5172,0.9375,0.6667,0.6964
9,1.0000,0.9583,0.9787,0.9789


In [31]:
class_ids = list(ogan_ilir_class.keys())
vis_params = {
    "min": min(class_ids),
    "max": max(class_ids),
    "palette": [ogan_ilir_class[i]["color"] for i in class_ids]
}

legend_dict = {
    ogan_ilir_class[i]["name"]: ogan_ilir_class[i]["color"]
    for i in class_ids
}
m = geemap.Map()
m.centerObject(aoi, 8)
m.addLayer(classification_raw, vis_params, "MS_Only_Classification")
m.addLayer(classification_embed, vis_params, "embedding classification")
m.add_legend(title="Land Cover", legend_dict=legend_dict)
m


Map(center=[-3.4152616959981676, 104.60534276364243], controls=(WidgetControl(options=['position', 'transparen…

In [32]:
export_task = ee.batch.Export.image.toDrive(
    image=classification_raw,
    description='Classification_ogan_MS',
    folder='Earth Engine',
    fileNamePrefix='Classification_ogan_MS',
    scale=30,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting.

In [ ]:
export_task = ee.batch.Export.image.toDrive(
    image=classification_embed,
    description='Classification_ogan_Embedding',
    folder='Earth Engine',
    fileNamePrefix='Classification_ogan_Embedding',
    scale=10,
    region=aoi.geometry(),  # or aoi.geometry()
    maxPixels=1e13
)
export_task.start()
import time

while export_task.active():
    print('Exporting... (status: {})'.format(export_task.status()['state']))
    time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
